# Canonical Correlation Analysis of 16S V1-V3 and V4 and metabolomics table with top VIPs

Date created: 1/29/2024

Notebook author: Yang Chen

Data analysis by: Britta de Pessemier, Yang Chen

In [2]:
import biom
from biom import load_table
from biom.util import biom_open
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import CCA
from sklearn.preprocessing import StandardScaler
from skbio.stats.composition import clr
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

In [3]:
# Function to load BIOM table, sort rows by sum
def biom_to_df(biom_path):

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)

    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])

    # Transpose df
    df = df.T
    
    return df


In [4]:
def save_as_biom(df: pd.DataFrame, output_path: str):
    """
    Save a pandas DataFrame as a BIOM table.

    Parameters:
    df (pd.DataFrame): The DataFrame to save.
    output_path (str): Path to the output BIOM file.
    """
    table = biom.table.Table(df.values, observation_ids=df.index, sample_ids=df.columns)
    with biom_open(output_path, 'w') as f:
        table.to_hdf5(f, "save biom")

In [5]:
# Read in the metadata
metadata = pd.read_csv('../Metadata/metadata_final_18062024.tsv', sep='\t')
metadata

,SampleID,c_zone,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face,zone,sample_type,planned_study_day_of_visit,visual_assessment_in_vivo_number_of_inflammatory_lesions_face,day,subject_randomization_number,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_cheek_right,...,inflammatory_lesions_face,noninflammatory_lesions_face,a,cohort,subject_randomization_id,class,subject_ID,subject_ID_CC,zone_CC,group
0,LAMI.RD308.D16.C1,C1,not applicable,Lesional,skin,Day 16,not applicable,16,308,not applicable,...,0,0,33.765638,acne,RD308,acne,PP_308,PP_308C1,Lesional_C1,Acne_L
1,LAMI.RD310.D21.C1,C1,72,Lesional,skin,Day 21,36,21,310,17,...,36,72,31.919478,acne,RD310,acne,PP_310,PP_310C1,Lesional_C1,Acne_L
2,LAMI.RD305.D21.C3,C3,69,Non-lesional,skin,Day 21,26,21,305,25,...,26,69,22.152503,acne,RD305,healthy,PP_305,PP_305C3,Non-lesional_C3,Acne_NL
3,LAMI.RD306.D18.C2,C2,not applicable,Lesional,skin,Day 18,not applicable,18,306,not applicable,...,0,0,27.397918,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L
4,LAMI.RD306.D7.C2,C2,90,Lesional,skin,Day 7,13,7,306,23,...,13,90,28.819451,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,LAMI.RD317.D21.C1,C1,77,Lesional,skin,Day 21,19,21,317,20,...,19,77,21.946648,acne,RD317,acne,PP_317,PP_317C1,Lesional_C1,Acne_L
262,LAMI.RD001.D0.C1,C1,not applicable,Non-lesional,skin,Day 0,not applicable,0,1,not applicable,...,0,0,26.344240,control,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy
263,LAMI.RD014.D14.C2,C2,not applicable,Non-lesional,skin,Day 14,not applicable,14,14,not applicable,...,0,0,16.359699,control,RD014,healthy,PP_14,PP_14C2,Non-lesional_C2,Healthy
264,LAMI.RD314.D0.C1,C1,55,Lesional,skin,Day 0,31,0,314,16,...,31,55,22.494605,acne,RD314,acne,PP_314,PP_314C1,Lesional_C1,Acne_L


In [6]:
# skin_group = "Acne_L"
# skin_group = "Acne_NL"
skin_group = "Healthy"

In [7]:
# Read in metabolomics table with top VIPs
metaB_tbl = pd.read_csv('../Data/metabolomics/Run3_10252024/output/metaB_top-VIPs_final.csv')
metaB_tbl = metaB_tbl.set_index('SampleID')

# Read in 16S V1-V3 table
V1V3_biom_path = '../Data/16S/Tables/16S_V1-V3_Genus_collapsed_absolute.biom'
V1V3_tbl = biom_to_df(V1V3_biom_path)

# Read in 16S V4 table
V4_biom_path = '../Data/16S/Tables/16S_V4_Genus_collapsed_absolute.biom'
V4_tbl = biom_to_df(V4_biom_path)

In [8]:
metaB_tbl

,Urocanic acid,(Iso)leucine,Phenylalanine,Tryptophan,Glutamine-C14:0,Sorbitane Monooleate,C19H36O4 Fatty alcohol
SampleID,,,,,,,
LAMI.RD001.D0.C1,39859.7580,2025607.40,970487.500,567510.060,0.0000,0.00,611709.700
LAMI.RD308.D2.C2,3065.1714,1669351.50,750870.100,442653.660,0.0000,0.00,231525.840
LAMI.RD304.D11.C1,36750.4000,1434033.20,595568.940,328151.120,5375.8525,99470.69,78491.530
LAMI.RD304.D11.C2,0.0000,706197.75,303845.300,152392.720,2691.0256,157503.92,261996.800
LAMI.RD304.D0.C1,3345.8184,778566.10,293386.340,137201.390,3431.0034,0.00,427202.060
...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,9787.1210,1250164.60,579058.800,498893.780,0.0000,0.00,122958.260
LAMI.RD308.D9.C3,3076.9688,687527.06,281896.700,262886.470,0.0000,0.00,65659.700
LAMI.RD319.D2.C2,0.0000,538412.94,255542.720,121803.890,0.0000,0.00,73231.560


In [9]:
# Subset metaB_tbl rows where sample is in V1V3_tbl
metaB_V1V3_tbl = metaB_tbl[metaB_tbl.index.isin(V1V3_tbl.index)]

# Remove the 'group' column (only for VIP table)
# metaB_V1V3_tbl = metaB_V1V3_tbl.drop(columns=['group'])
metaB_V1V3_tbl.index.name = None
metaB_V1V3_tbl

,Urocanic acid,(Iso)leucine,Phenylalanine,Tryptophan,Glutamine-C14:0,Sorbitane Monooleate,C19H36O4 Fatty alcohol
LAMI.RD001.D0.C1,39859.7580,2025607.40,970487.500,567510.060,0.0000,0.000,611709.700
LAMI.RD304.D11.C1,36750.4000,1434033.20,595568.940,328151.120,5375.8525,99470.690,78491.530
LAMI.RD304.D11.C2,0.0000,706197.75,303845.300,152392.720,2691.0256,157503.920,261996.800
LAMI.RD304.D0.C1,3345.8184,778566.10,293386.340,137201.390,3431.0034,0.000,427202.060
LAMI.RD304.D0.C2,24984.3220,2486485.20,1051277.400,459814.620,13703.5760,62253.426,223301.700
...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,9787.1210,1250164.60,579058.800,498893.780,0.0000,0.000,122958.260
LAMI.RD308.D9.C3,3076.9688,687527.06,281896.700,262886.470,0.0000,0.000,65659.700
LAMI.RD319.D2.C2,0.0000,538412.94,255542.720,121803.890,0.0000,0.000,73231.560
LAMI.RD319.D2.C3,1407.5230,106901.95,59064.336,23485.404,0.0000,0.000,55986.290


In [10]:
# Convert to relative abundance (row-wise normalization)
metaB_V1V3_tbl_relative = metaB_V1V3_tbl.div(metaB_V1V3_tbl.sum(axis=1), axis=0)
metaB_V1V3_tbl_relative

,Urocanic acid,(Iso)leucine,Phenylalanine,Tryptophan,Glutamine-C14:0,Sorbitane Monooleate,C19H36O4 Fatty alcohol
LAMI.RD001.D0.C1,0.009456,0.480551,0.230237,0.134635,0.000000,0.000000,0.145121
LAMI.RD304.D11.C1,0.014256,0.556292,0.231034,0.127297,0.002085,0.038587,0.030449
LAMI.RD304.D11.C2,0.000000,0.445655,0.191746,0.096169,0.001698,0.099395,0.165337
LAMI.RD304.D0.C1,0.002036,0.473830,0.178553,0.083500,0.002088,0.000000,0.259992
LAMI.RD304.D0.C2,0.005781,0.575333,0.243249,0.106394,0.003171,0.014404,0.051668
...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,0.003977,0.508019,0.235307,0.202731,0.000000,0.000000,0.049966
LAMI.RD308.D9.C3,0.002365,0.528441,0.216669,0.202058,0.000000,0.000000,0.050467
LAMI.RD319.D2.C2,0.000000,0.544406,0.258387,0.123160,0.000000,0.000000,0.074047
LAMI.RD319.D2.C3,0.005702,0.433072,0.239277,0.095142,0.000000,0.000000,0.226807


In [11]:
# Filter metadata for the correct group
filtered_metadata = metadata[metadata['group'] == skin_group]

# Normalize SampleID and index formatting
metaB_V1V3_tbl_relative.index = metaB_V1V3_tbl_relative.index.str.strip().str.upper()
filtered_metadata['SampleID'] = filtered_metadata['SampleID'].str.strip().str.upper()

# Filter metadata to only include matching SampleIDs
filtered_metadata = filtered_metadata[filtered_metadata['SampleID'].isin(metaB_V1V3_tbl_relative.index)]

# Subset the DataFrame
metaB_V1V3_tbl_relative = metaB_V1V3_tbl_relative.loc[filtered_metadata['SampleID']]

metaB_V1V3_tbl_relative

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_77620/3342871550.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_metadata['SampleID'] = filtered_metadata['SampleID'].str.strip().str.upper()


,Urocanic acid,(Iso)leucine,Phenylalanine,Tryptophan,Glutamine-C14:0,Sorbitane Monooleate,C19H36O4 Fatty alcohol
LAMI.RD002.D14.C1,0.001925,0.475034,0.241888,0.102212,0.0,0.000000,0.178941
LAMI.RD003.D14.C1,0.000000,0.609399,0.249369,0.115947,0.0,0.000000,0.025286
LAMI.RD017.D0.C2,0.002547,0.500017,0.246914,0.144022,0.0,0.010050,0.096450
LAMI.RD017.D28.C2,0.000523,0.526244,0.261695,0.158392,0.0,0.003088,0.050059
LAMI.RD004.D0.C1,0.005122,0.418503,0.170999,0.126074,0.0,0.019134,0.260167
LAMI.RD003.D28.C1,0.004022,0.582325,0.234826,0.158616,0.0,0.000000,0.020210
LAMI.RD001.D14.C1,0.005206,0.493885,0.224242,0.105805,0.0,0.000000,0.170861
LAMI.RD001.D28.C1,0.007567,0.495924,0.262576,0.189465,0.0,0.005265,0.039202
LAMI.RD002.D14.C2,0.000000,0.573687,0.261463,0.100725,0.0,0.000000,0.064125
LAMI.RD006.D14.C1,0.005587,0.615529,0.236214,0.137572,0.0,0.000000,0.005098


In [12]:
# Save as biom for mmvec
metaB_V1V3_tbl_transposed = metaB_V1V3_tbl_relative.T
output_path = f'../Data/multi-omics/metabolomics-tbl_16S_V1V3-matched_relative_{skin_group}.biom'
save_as_biom(metaB_V1V3_tbl_transposed, output_path)

In [13]:
# Subset V1V3_tbl rows where sample is in metaB_tbl
V1V3_tbl_matched = V1V3_tbl[V1V3_tbl.index.isin(metaB_V1V3_tbl_relative.index)]

# top_V1V3_features = [' g__Cutibacterium', ' g__Staphylococcus',
#                      ' g__Streptococcus', ' g__Corynebacterium', ' g__Lawsonella',
#                      ' g__Micrococcus', ' g__Alloprevotella', ' g__Lactobacillus', 
#                      ' g__Rothia', ' g__Kocuria', ' g__Haemophilus']

top_V1V3_features = [' g__Cutibacterium', ' g__Staphylococcus',
                     ' g__Streptococcus',' g__Lawsonella',
                     ' g__Micrococcus', ' g__Lactobacillus', 
                     ' g__Rothia', ' g__Kocuria', ' g__Haemophilus', ' g__Corynebacterium']

# Find the intersection of desired columns and actual DataFrame columns
available_columns = V1V3_tbl_matched.columns.intersection(top_V1V3_features)
V1V3_tbl_matched = V1V3_tbl_matched[available_columns]

V1V3_tbl_matched

,g__Cutibacterium,g__Staphylococcus,g__Streptococcus,g__Corynebacterium,g__Lawsonella,g__Micrococcus,g__Lactobacillus,g__Rothia,g__Haemophilus,g__Kocuria
LAMI.RD001.D0.C1,2362.0,94.0,292.0,175.0,91.0,37.0,1.0,50.0,87.0,0.0
LAMI.RD001.D14.C1,2234.0,83.0,240.0,492.0,253.0,30.0,60.0,12.0,8.0,0.0
LAMI.RD001.D14.C2,1761.0,137.0,446.0,456.0,151.0,16.0,26.0,31.0,58.0,6.0
LAMI.RD001.D28.C1,2367.0,118.0,293.0,365.0,217.0,22.0,17.0,14.0,10.0,5.0
LAMI.RD002.D0.C2,2900.0,373.0,159.0,39.0,13.0,61.0,1.0,27.0,11.0,54.0
LAMI.RD003.D14.C1,3296.0,95.0,14.0,138.0,14.0,0.0,1.0,5.0,0.0,0.0
LAMI.RD002.D14.C1,3440.0,105.0,43.0,49.0,41.0,7.0,2.0,4.0,1.0,0.0
LAMI.RD003.D28.C1,2367.0,103.0,543.0,31.0,36.0,0.0,0.0,16.0,21.0,0.0
LAMI.RD007.D0.C1,3707.0,23.0,16.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD002.D14.C2,3092.0,96.0,57.0,34.0,33.0,10.0,0.0,246.0,1.0,17.0


In [14]:
# Convert to relative abundance (row-wise normalization)
V1V3_tbl_matched_relative = V1V3_tbl_matched.div(V1V3_tbl_matched.sum(axis=1), axis=0)
V1V3_tbl_matched_relative

,g__Cutibacterium,g__Staphylococcus,g__Streptococcus,g__Corynebacterium,g__Lawsonella,g__Micrococcus,g__Lactobacillus,g__Rothia,g__Haemophilus,g__Kocuria
LAMI.RD001.D0.C1,0.740671,0.029476,0.091565,0.054876,0.028536,0.011602,0.000314,0.015679,0.027281,0.000000
LAMI.RD001.D14.C1,0.654748,0.024326,0.070340,0.144197,0.074150,0.008792,0.017585,0.003517,0.002345,0.000000
LAMI.RD001.D14.C2,0.570272,0.044365,0.144430,0.147668,0.048899,0.005181,0.008420,0.010039,0.018782,0.001943
LAMI.RD001.D28.C1,0.690490,0.034422,0.085473,0.106476,0.063302,0.006418,0.004959,0.004084,0.002917,0.001459
LAMI.RD002.D0.C2,0.797141,0.102529,0.043705,0.010720,0.003573,0.016767,0.000275,0.007422,0.003024,0.014843
LAMI.RD003.D14.C1,0.925063,0.026663,0.003929,0.038731,0.003929,0.000000,0.000281,0.001403,0.000000,0.000000
LAMI.RD002.D14.C1,0.931744,0.028440,0.011647,0.013272,0.011105,0.001896,0.000542,0.001083,0.000271,0.000000
LAMI.RD003.D28.C1,0.759384,0.033045,0.174206,0.009945,0.011550,0.000000,0.000000,0.005133,0.006737,0.000000
LAMI.RD007.D0.C1,0.987743,0.006128,0.004263,0.000000,0.001865,0.000000,0.000000,0.000000,0.000000,0.000000
LAMI.RD002.D14.C2,0.862242,0.026771,0.015895,0.009481,0.009202,0.002789,0.000000,0.068600,0.000279,0.004741


In [15]:
# Save metadata for V1V3
metadata_V1V3 = metadata[metadata['SampleID'].isin(V1V3_tbl_matched_relative.index)]
metadata_V1V3 = metadata_V1V3[['SampleID', 'group']]

# Rename the first column to 'sample id'
metadata_V1V3.rename(columns={metadata_V1V3.columns[0]: 'sample id'}, inplace=True)

# Remove the name of the second column (set it to an empty string)
metadata_V1V3.columns = ['sample id'] + ['' if i == 1 else col for i, col in enumerate(metadata_V1V3.columns[1:])]

metadata_V1V3.to_csv('../Data/multi-omics/16S_V1V3_metaB-matched.csv', index=0)
metadata_V1V3

,sample id,group
9,LAMI.RD002.D14.C1,Healthy
14,LAMI.RD003.D14.C1,Healthy
22,LAMI.RD017.D0.C2,Healthy
28,LAMI.RD017.D28.C2,Healthy
33,LAMI.RD004.D0.C1,Healthy
36,LAMI.RD003.D28.C1,Healthy
38,LAMI.RD001.D14.C1,Healthy
45,LAMI.RD001.D28.C1,Healthy
54,LAMI.RD002.D14.C2,Healthy
58,LAMI.RD006.D14.C1,Healthy


In [16]:
# Save as biom for mmvec
V1V3_tbl_matched_transposed = V1V3_tbl_matched_relative.T
output_path = f'../Data/multi-omics/16S_V1V3-tbl_metaB-matched_relative_{skin_group}.biom'
save_as_biom(V1V3_tbl_matched_transposed, output_path)

In [17]:
# Create column_names_df with 'feature id' as the column header
column_names_df = pd.DataFrame({"feature id": metaB_V1V3_tbl.columns})  # Ensure feature id is a string

column_names_df

,feature id
0,Urocanic acid
1,(Iso)leucine
2,Phenylalanine
3,Tryptophan
4,Glutamine-C14:0
5,Sorbitane Monooleate
6,C19H36O4 Fatty alcohol


In [18]:
# Save to a CSV file
output_path = "../Data/multi-omics/metabolite_info.csv"
column_names_df = column_names_df.to_csv(output_path, index=0)

In [19]:
# Subset metaB rows where sample is in V4_tbl
metaB_V4_tbl = metaB_tbl[metaB_tbl.index.isin(V4_tbl.index)]
# metaB_V4_tbl = metaB_V4_tbl.drop('group', axis=1)
metaB_V4_tbl.index.name = None
metaB_V4_tbl

,Urocanic acid,(Iso)leucine,Phenylalanine,Tryptophan,Glutamine-C14:0,Sorbitane Monooleate,C19H36O4 Fatty alcohol
LAMI.RD001.D0.C1,39859.7580,2025607.40,970487.500,567510.060,0.0000,0.000,611709.700
LAMI.RD304.D11.C1,36750.4000,1434033.20,595568.940,328151.120,5375.8525,99470.690,78491.530
LAMI.RD304.D11.C2,0.0000,706197.75,303845.300,152392.720,2691.0256,157503.920,261996.800
LAMI.RD304.D0.C1,3345.8184,778566.10,293386.340,137201.390,3431.0034,0.000,427202.060
LAMI.RD304.D0.C2,24984.3220,2486485.20,1051277.400,459814.620,13703.5760,62253.426,223301.700
...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,9787.1210,1250164.60,579058.800,498893.780,0.0000,0.000,122958.260
LAMI.RD308.D9.C3,3076.9688,687527.06,281896.700,262886.470,0.0000,0.000,65659.700
LAMI.RD319.D2.C2,0.0000,538412.94,255542.720,121803.890,0.0000,0.000,73231.560
LAMI.RD319.D2.C3,1407.5230,106901.95,59064.336,23485.404,0.0000,0.000,55986.290


In [20]:
# Drop the column from the DataFrame as it wasn't significant after univariate testing
# metaB_V4_tbl = metaB_V4_tbl.drop(columns=['C19H22O4 Linear diarylheptanoids'])

In [21]:
# Convert to relative abundance (row-wise normalization)
metaB_V4_tbl_relative = metaB_V4_tbl.div(metaB_V4_tbl.sum(axis=1), axis=0)
metaB_V4_tbl_relative

,Urocanic acid,(Iso)leucine,Phenylalanine,Tryptophan,Glutamine-C14:0,Sorbitane Monooleate,C19H36O4 Fatty alcohol
LAMI.RD001.D0.C1,0.009456,0.480551,0.230237,0.134635,0.000000,0.000000,0.145121
LAMI.RD304.D11.C1,0.014256,0.556292,0.231034,0.127297,0.002085,0.038587,0.030449
LAMI.RD304.D11.C2,0.000000,0.445655,0.191746,0.096169,0.001698,0.099395,0.165337
LAMI.RD304.D0.C1,0.002036,0.473830,0.178553,0.083500,0.002088,0.000000,0.259992
LAMI.RD304.D0.C2,0.005781,0.575333,0.243249,0.106394,0.003171,0.014404,0.051668
...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,0.003977,0.508019,0.235307,0.202731,0.000000,0.000000,0.049966
LAMI.RD308.D9.C3,0.002365,0.528441,0.216669,0.202058,0.000000,0.000000,0.050467
LAMI.RD319.D2.C2,0.000000,0.544406,0.258387,0.123160,0.000000,0.000000,0.074047
LAMI.RD319.D2.C3,0.005702,0.433072,0.239277,0.095142,0.000000,0.000000,0.226807


In [22]:
# Filter metadata for the correct group
filtered_metadata = metadata[metadata['group'] == skin_group]

# Normalize SampleID and index formatting
metaB_V4_tbl_relative.index = metaB_V4_tbl_relative.index.str.strip().str.upper()
filtered_metadata['SampleID'] = filtered_metadata['SampleID'].str.strip().str.upper()

# Filter metadata to only include matching SampleIDs
filtered_metadata = filtered_metadata[filtered_metadata['SampleID'].isin(metaB_V4_tbl_relative.index)]

# Subset the DataFrame
metaB_V4_tbl_relative = metaB_V4_tbl_relative.loc[filtered_metadata['SampleID']]

metaB_V4_tbl_relative

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_77620/4148729606.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_metadata['SampleID'] = filtered_metadata['SampleID'].str.strip().str.upper()


,Urocanic acid,(Iso)leucine,Phenylalanine,Tryptophan,Glutamine-C14:0,Sorbitane Monooleate,C19H36O4 Fatty alcohol
LAMI.RD017.D0.C1,0.001954,0.484408,0.256801,0.168835,0.001089,0.010848,0.076066
LAMI.RD017.D0.C2,0.002547,0.500017,0.246914,0.144022,0.000000,0.010050,0.096450
LAMI.RD017.D28.C2,0.000523,0.526244,0.261695,0.158392,0.000000,0.003088,0.050059
LAMI.RD004.D0.C1,0.005122,0.418503,0.170999,0.126074,0.000000,0.019134,0.260167
LAMI.RD001.D14.C1,0.005206,0.493885,0.224242,0.105805,0.000000,0.000000,0.170861
LAMI.RD001.D28.C1,0.007567,0.495924,0.262576,0.189465,0.000000,0.005265,0.039202
LAMI.RD017.D28.C1,0.000643,0.532446,0.249223,0.157582,0.000000,0.003895,0.056212
LAMI.RD017.D14.C2,0.001256,0.554394,0.239752,0.137031,0.000672,0.001788,0.065106
LAMI.RD006.D14.C1,0.005587,0.615529,0.236214,0.137572,0.000000,0.000000,0.005098
LAMI.RD018.D0.C2,0.023259,0.466531,0.199136,0.126619,0.000000,0.000000,0.184456


In [23]:
# Subset V4_tbl rows where sample is in metaB_tbl
V4_tbl_matched = V4_tbl[V4_tbl.index.isin(metaB_V4_tbl_relative.index)]

# top_V4_features = [' g__Staphylococcus', ' g__Lawsonella',
#                     ' g__Streptococcus', ' g__Acinetobacter', ' g__Cutibacterium', 'g__Enhydrobacter',
#                      ' g__Micrococcus', ' g__Lactobacillus', ' g__Rothia', ' g__Kocuria', ' g__Haemophilus', ' g__Corynebacterium']


top_V4_features = [' g__Staphylococcus', ' g__Lawsonella', ' g__Streptococcus', ' g__Cutibacterium',
                     ' g__Micrococcus', ' g__Lactobacillus', ' g__Rothia', ' g__Kocuria', ' g__Haemophilus', ' g__Corynebacterium']

# Find the intersection of desired columns and actual DataFrame columns
available_columns = V4_tbl_matched.columns.intersection(top_V4_features)
V4_tbl_matched = V4_tbl_matched[available_columns]

V4_tbl_matched

,g__Staphylococcus,g__Lawsonella,g__Corynebacterium,g__Streptococcus,g__Haemophilus,g__Micrococcus,g__Lactobacillus,g__Cutibacterium,g__Kocuria,g__Rothia
LAMI.RD001.D0.C1,317.0,131.0,275.0,414.0,475.0,100.0,15.0,1.0,17.0,36.0
LAMI.RD001.D14.C1,454.0,456.0,915.0,247.0,168.0,93.0,117.0,0.0,5.0,15.0
LAMI.RD001.D14.C2,444.0,206.0,546.0,457.0,456.0,54.0,23.0,0.0,5.0,47.0
LAMI.RD017.D14.C2,1756.0,636.0,131.0,33.0,25.0,617.0,2.0,126.0,7.0,1.0
LAMI.RD011.D14.C1,2063.0,999.0,102.0,85.0,37.0,88.0,45.0,32.0,3.0,7.0
LAMI.RD004.D0.C1,603.0,48.0,875.0,270.0,14.0,83.0,21.0,6.0,5.0,30.0
LAMI.RD001.D28.C1,580.0,457.0,692.0,349.0,142.0,72.0,83.0,1.0,5.0,34.0
LAMI.RD017.D28.C1,1189.0,268.0,120.0,202.0,95.0,237.0,14.0,69.0,5.0,2.0
LAMI.RD011.D14.C2,567.0,112.0,59.0,15.0,2.0,0.0,6.0,12.0,6.0,1.0
LAMI.RD017.D0.C1,2876.0,168.0,119.0,88.0,23.0,181.0,0.0,65.0,2.0,0.0


In [24]:
# Convert to relative abundance (row-wise normalization)
V4_tbl_matched_relative = V4_tbl_matched.div(V4_tbl_matched.sum(axis=1), axis=0)
V4_tbl_matched_relative

,g__Staphylococcus,g__Lawsonella,g__Corynebacterium,g__Streptococcus,g__Haemophilus,g__Micrococcus,g__Lactobacillus,g__Cutibacterium,g__Kocuria,g__Rothia
LAMI.RD001.D0.C1,0.177990,0.073554,0.154408,0.232454,0.266704,0.056148,0.008422,0.000561,0.009545,0.020213
LAMI.RD001.D14.C1,0.183806,0.184615,0.370445,0.100000,0.068016,0.037652,0.047368,0.000000,0.002024,0.006073
LAMI.RD001.D14.C2,0.198391,0.092046,0.243968,0.204200,0.203753,0.024129,0.010277,0.000000,0.002234,0.021001
LAMI.RD017.D14.C2,0.526695,0.190762,0.039292,0.009898,0.007499,0.185063,0.000600,0.037792,0.002100,0.000300
LAMI.RD011.D14.C1,0.596070,0.288645,0.029471,0.024559,0.010691,0.025426,0.013002,0.009246,0.000867,0.002023
LAMI.RD004.D0.C1,0.308440,0.024552,0.447570,0.138107,0.007161,0.042455,0.010742,0.003069,0.002558,0.015345
LAMI.RD001.D28.C1,0.240166,0.189234,0.286542,0.144513,0.058799,0.029814,0.034369,0.000414,0.002070,0.014079
LAMI.RD017.D28.C1,0.540209,0.121763,0.054521,0.091776,0.043162,0.107678,0.006361,0.031349,0.002272,0.000909
LAMI.RD011.D14.C2,0.726923,0.143590,0.075641,0.019231,0.002564,0.000000,0.007692,0.015385,0.007692,0.001282
LAMI.RD017.D0.C1,0.816581,0.047700,0.033788,0.024986,0.006530,0.051391,0.000000,0.018455,0.000568,0.000000


In [25]:
# Save metadata for V4
metadata_V4 = metadata[metadata['SampleID'].isin(V4_tbl_matched.index)]
metadata_V4 = metadata_V4[['SampleID', 'group']]
# Rename the first column to 'sample id'
metadata_V4.rename(columns={metadata_V4.columns[0]: 'sample id'}, inplace=True)

# Remove the name of the second column (set it to an empty string)
metadata_V4.columns = ['sample id'] + ['' if i == 1 else col for i, col in enumerate(metadata_V4.columns[1:])]

metadata_V4.to_csv('../Data/multi-omics/16S_V4_metaB-matched.csv', index=0)
metadata_V4

,sample id,group
18,LAMI.RD017.D0.C1,Healthy
22,LAMI.RD017.D0.C2,Healthy
28,LAMI.RD017.D28.C2,Healthy
33,LAMI.RD004.D0.C1,Healthy
38,LAMI.RD001.D14.C1,Healthy
45,LAMI.RD001.D28.C1,Healthy
46,LAMI.RD017.D28.C1,Healthy
49,LAMI.RD017.D14.C2,Healthy
58,LAMI.RD006.D14.C1,Healthy
66,LAMI.RD018.D0.C2,Healthy


In [26]:
# Save as biom for mmvec
metaB_V4_tbl_transposed = metaB_V4_tbl_relative.T
output_path = f'../Data/multi-omics/metabolomics-tbl_16S_V4-matched_relative_{skin_group}.biom'
save_as_biom(metaB_V4_tbl_transposed, output_path)

In [27]:
# Save as biom for mmvec
V4_tbl_matched_transposed = V4_tbl_matched_relative.T
output_path = '../Data/multi-omics/16S_V4-tbl_metaB-matched_relative.biom'
save_as_biom(V4_tbl_matched_transposed, output_path)

### Combine V1-V3 and V4 tables

In [28]:
# # Find common rows (samples)
# common_rows = V1V3_tbl_matched.index.intersection(V4_tbl_matched.index)

# # Subset both DataFrames to keep only common rows
# V1V3_tbl_matched = V1V3_tbl_matched.loc[common_rows]
# V4_tbl_matched = V4_tbl_matched.loc[common_rows]

# # Merge by taking average of values row-wise
# merged_tbl = V1V3_tbl_matched.add(V4_tbl_matched, fill_value=0)

# # Sort columns by total sum (descending order)
# merged_tbl = merged_tbl[merged_tbl.sum().sort_values(ascending=False).index]

# merged_tbl

In [29]:
V1V3_tbl_matched.columns

Index([' g__Cutibacterium', ' g__Staphylococcus', ' g__Streptococcus',
       ' g__Corynebacterium', ' g__Lawsonella', ' g__Micrococcus',
       ' g__Lactobacillus', ' g__Rothia', ' g__Haemophilus', ' g__Kocuria'],
      dtype='object')

In [30]:
V4_tbl_matched.columns

Index([' g__Staphylococcus', ' g__Lawsonella', ' g__Corynebacterium',
       ' g__Streptococcus', ' g__Haemophilus', ' g__Micrococcus',
       ' g__Lactobacillus', ' g__Cutibacterium', ' g__Kocuria', ' g__Rothia'],
      dtype='object')

In [31]:
# Find common rows (samples)
common_rows = V1V3_tbl_matched.index.intersection(V4_tbl_matched.index)

# Subset both DataFrames to keep only common rows
V1V3_tbl_matched = V1V3_tbl_matched.loc[common_rows]
V4_tbl_matched = V4_tbl_matched.loc[common_rows]

# Compute the mean for each matching row and column
merged_tbl = (V1V3_tbl_matched + V4_tbl_matched) / 2

# Sort columns by total mean (descending order)
merged_tbl = merged_tbl[merged_tbl.mean().sort_values(ascending=False).index]

# Display merged DataFrame
merged_tbl


,g__Cutibacterium,g__Staphylococcus,g__Corynebacterium,g__Streptococcus,g__Lawsonella,g__Micrococcus,g__Haemophilus,g__Kocuria,g__Lactobacillus,g__Rothia
LAMI.RD001.D0.C1,1181.5,205.5,225.0,353.0,111.0,68.5,281.0,8.5,8.0,43.0
LAMI.RD001.D14.C1,1117.0,268.5,703.5,243.5,354.5,61.5,88.0,2.5,88.5,13.5
LAMI.RD001.D14.C2,880.5,290.5,501.0,451.5,178.5,35.0,257.0,5.5,24.5,39.0
LAMI.RD001.D28.C1,1184.0,349.0,528.5,321.0,337.0,47.0,76.0,5.0,50.0,24.0
LAMI.RD004.D0.C1,1340.0,352.5,664.5,290.0,35.0,52.0,8.5,2.5,16.0,29.5
LAMI.RD006.D14.C2,1700.0,387.0,458.5,65.5,208.0,78.0,7.5,16.5,66.0,1.5
LAMI.RD017.D0.C2,1905.5,990.5,148.0,48.0,160.0,261.0,47.0,0.0,0.0,0.0
LAMI.RD007.D28.C1,1856.5,1348.5,100.0,93.0,70.5,7.5,46.5,0.5,12.5,0.0
LAMI.RD006.D14.C1,1346.5,269.0,409.5,382.5,141.5,72.0,3.5,40.0,125.0,22.5
LAMI.RD007.D28.C2,1860.5,880.5,275.5,21.5,198.5,7.5,69.0,0.5,15.0,1.5


In [32]:
# Convert to relative abundance (row-wise normalization)
merged_tbl_relative = merged_tbl.div(merged_tbl.sum(axis=1), axis=0)
merged_tbl_relative

,g__Cutibacterium,g__Staphylococcus,g__Corynebacterium,g__Streptococcus,g__Lawsonella,g__Micrococcus,g__Haemophilus,g__Kocuria,g__Lactobacillus,g__Rothia
LAMI.RD001.D0.C1,0.475453,0.082696,0.090543,0.142052,0.044668,0.027565,0.113078,0.003421,0.003219,0.017304
LAMI.RD001.D14.C1,0.379803,0.091295,0.239204,0.082795,0.120537,0.020911,0.029922,0.000850,0.030092,0.004590
LAMI.RD001.D14.C2,0.330642,0.109087,0.188134,0.169546,0.067030,0.013143,0.096508,0.002065,0.009200,0.014645
LAMI.RD001.D28.C1,0.405271,0.119459,0.180900,0.109875,0.115352,0.016088,0.026014,0.001711,0.017114,0.008215
LAMI.RD004.D0.C1,0.480201,0.126321,0.238129,0.103924,0.012543,0.018635,0.003046,0.000896,0.005734,0.010572
LAMI.RD006.D14.C2,0.568847,0.129496,0.153421,0.021917,0.069600,0.026100,0.002510,0.005521,0.022085,0.000502
LAMI.RD017.D0.C2,0.535253,0.278230,0.041573,0.013483,0.044944,0.073315,0.013202,0.000000,0.000000,0.000000
LAMI.RD007.D28.C1,0.525103,0.381417,0.028285,0.026305,0.019941,0.002121,0.013152,0.000141,0.003536,0.000000
LAMI.RD006.D14.C1,0.478841,0.095661,0.145626,0.136024,0.050320,0.025605,0.001245,0.014225,0.044452,0.008001
LAMI.RD007.D28.C2,0.558709,0.264414,0.082733,0.006456,0.059610,0.002252,0.020721,0.000150,0.004505,0.000450


In [33]:
merged_tbl_relative.columns

Index([' g__Cutibacterium', ' g__Staphylococcus', ' g__Corynebacterium',
       ' g__Streptococcus', ' g__Lawsonella', ' g__Micrococcus',
       ' g__Haemophilus', ' g__Kocuria', ' g__Lactobacillus', ' g__Rothia'],
      dtype='object')

In [34]:
# Save as biom for mmvec
merged_tbl_relative_transposed = merged_tbl_relative.T
output_path = f'../Data/multi-omics/16S_MERGED-Avg-tbl_metaB-matched_relative_{skin_group}.biom'
save_as_biom(merged_tbl_relative_transposed, output_path)

In [35]:
# Subset metaB_tbl rows where sample is in V1V3_tbl
metaB_MERGED_tbl = metaB_tbl[metaB_tbl.index.isin(merged_tbl_relative.index)]

# Remove the 'group' column (only for VIP table)
# metaB_MERGED_tbl = metaB_MERGED_tbl.drop(columns=['group'])
metaB_MERGED_tbl.index.name = None

In [36]:
# Sort columns by total sum (descending order)
metaB_MERGED_tbl = metaB_MERGED_tbl[metaB_MERGED_tbl.sum().sort_values(ascending=False).index]
metaB_MERGED_tbl

,(Iso)leucine,Phenylalanine,Tryptophan,C19H36O4 Fatty alcohol,Urocanic acid,Sorbitane Monooleate,Glutamine-C14:0
LAMI.RD001.D0.C1,2025607.40,970487.50,567510.06,611709.700,39859.7580,0.000,0.0
LAMI.RD001.D14.C1,3296738.00,1496841.80,706262.44,1140515.800,34747.9200,0.000,0.0
LAMI.RD001.D14.C2,3165493.20,1258087.60,717058.00,263219.340,0.0000,0.000,0.0
LAMI.RD001.D28.C1,3403914.80,1802260.60,1300446.40,269075.660,51941.2540,36139.850,0.0
LAMI.RD006.D14.C1,873745.75,335306.12,195284.60,7236.388,7931.4785,0.000,0.0
LAMI.RD006.D14.C2,1288035.90,547077.70,332766.40,15614.556,25884.8440,11288.382,0.0
LAMI.RD007.D28.C1,645682.70,272316.44,97546.94,885806.000,0.0000,0.000,0.0
LAMI.RD007.D28.C2,902520.06,421043.12,229239.88,344229.940,2022.2802,43746.200,0.0
LAMI.RD017.D0.C2,2193938.80,1083391.00,631929.06,423195.160,11175.1790,44098.290,0.0
LAMI.RD011.D14.C2,785636.30,305970.00,162636.17,27341.186,0.0000,0.000,0.0


In [37]:
# Convert to relative abundance (row-wise normalization)
metaB_MERGED_tbl_relative = metaB_MERGED_tbl.div(metaB_MERGED_tbl.sum(axis=1), axis=0)
metaB_MERGED_tbl_relative

,(Iso)leucine,Phenylalanine,Tryptophan,C19H36O4 Fatty alcohol,Urocanic acid,Sorbitane Monooleate,Glutamine-C14:0
LAMI.RD001.D0.C1,0.480551,0.230237,0.134635,0.145121,0.009456,0.000000,0.0
LAMI.RD001.D14.C1,0.493885,0.224242,0.105805,0.170861,0.005206,0.000000,0.0
LAMI.RD001.D14.C2,0.585784,0.232813,0.132694,0.048710,0.000000,0.000000,0.0
LAMI.RD001.D28.C1,0.495924,0.262576,0.189465,0.039202,0.007567,0.005265,0.0
LAMI.RD006.D14.C1,0.615529,0.236214,0.137572,0.005098,0.005587,0.000000,0.0
LAMI.RD006.D14.C2,0.580022,0.246357,0.149850,0.007031,0.011656,0.005083,0.0
LAMI.RD007.D28.C1,0.339591,0.143223,0.051304,0.465882,0.000000,0.000000,0.0
LAMI.RD007.D28.C2,0.464546,0.216720,0.117994,0.177182,0.001041,0.022517,0.0
LAMI.RD017.D0.C2,0.500017,0.246914,0.144022,0.096450,0.002547,0.010050,0.0
LAMI.RD011.D14.C2,0.613020,0.238744,0.126903,0.021334,0.000000,0.000000,0.0


In [38]:
# Save as biom for mmvec
metaB_MERGED_tbl_relative_transposed = metaB_MERGED_tbl_relative.T
output_path = f'../Data/multi-omics/metaB_MERGED-Avg-tbl_matched_relative_{skin_group}.biom'
save_as_biom(metaB_MERGED_tbl_relative_transposed, output_path)

In [39]:
metaB_MERGED_tbl_relative.columns

Index(['(Iso)leucine', 'Phenylalanine', 'Tryptophan', 'C19H36O4 Fatty alcohol',
       'Urocanic acid', 'Sorbitane Monooleate', 'Glutamine-C14:0'],
      dtype='object')